# BÀI TẬP CHƯƠNG 2

Khởi tạo và chuẩn bị dữ liệu bằng **Pandas** và **SQLite** (hoặc `pandasql`/`duckdb`), ở đây ta dùng thư viện `sqlite3` tích hợp sẵn cùng `pandas` để trực quan hiển thị dạng bảng.

In [ ]:
import pandas as pd
import sqlite3
import datetime

# Kết nối (hoặc tạo) cơ sở dữ liệu trên RAM
conn = sqlite3.connect(':memory:')

# Tạo bảng student và course
conn.executescript('''
CREATE TABLE student (
    student_id INTEGER PRIMARY KEY,
    name TEXT,
    class TEXT,
    course_id INTEGER,
    score REAL
);

CREATE TABLE course (
    id INTEGER PRIMARY KEY,
    course_name TEXT
);

INSERT INTO student (student_id, name, class, course_id, score) VALUES
(1, 'Nguyen Minh Hoang', 'May Tinh', 12, 6.7),
(2, 'Tran Thi Lan', 'Kinh Te', 34, 9.2),
(3, 'Pham Van Nam', 'Toan Tin', NULL, 7.9),
(4, 'Le Thanh Huyen', 'Toan Tin', 20, 7.2),
(5, 'Vu Quoc Anh', 'May Tinh', 24, 8.0),
(6, 'Dang Thuy Linh', 'May Tinh', 24, 5.5),
(7, 'Bui Tien Dung', 'Kinh Te', 34, 9.2),
(8, 'Ho Ngoc Mai', 'Toan Tin', 20, 8.8),
(9, 'Duong Huu Phuc', 'Kinh Te', NULL, 7.2),
(10, 'Cao Thi Hanh', 'May Tinh', NULL, 7.0);

INSERT INTO course (id, course_name) VALUES
(12, 'Giai tich'),
(34, 'Thong ke'),
(26, 'Tin hoc');
''')

print("Khởi tạo dữ liệu thành công!")


---

### 1. Kết nối hai bảng
**1.1. Tích Descartes** (CROSS JOIN)

In [ ]:
query_cross = "SELECT * FROM student CROSS JOIN course;"
df_cross = pd.read_sql_query(query_cross, conn)
display(df_cross)


**1.2. INNER JOIN, LEFT JOIN, RIGHT JOIN, FULL OUTER JOIN**

In [ ]:
print("--- INNER JOIN ---")
query_inner = "SELECT * FROM student INNER JOIN course ON student.course_id = course.id;"
display(pd.read_sql_query(query_inner, conn))

print("--- LEFT JOIN ---")
query_left = "SELECT * FROM student LEFT JOIN course ON student.course_id = course.id;"
display(pd.read_sql_query(query_left, conn))

print("--- RIGHT JOIN ---")
# Trong sqlite hiện tại có thể dùng RIGHT JOIN, nếu không hỗ trợ thì đổi chỗ LEFT JOIN
query_right = "SELECT * FROM student RIGHT JOIN course ON student.course_id = course.id;"
try:
    display(pd.read_sql_query(query_right, conn))
except Exception as e:
    # Nếu bản sqlite cũ không hỗ trợ right join
    query_right_alt = "SELECT * FROM course LEFT JOIN student ON course.id = student.course_id;"
    display(pd.read_sql_query(query_right_alt, conn))

print("--- FULL OUTER JOIN ---")
try:
    query_full = "SELECT * FROM student FULL OUTER JOIN course ON student.course_id = course.id;"
    display(pd.read_sql_query(query_full, conn))
except Exception as e:
    print("Bản SQLite hiện hành không hỗ trợ FULL OUTER JOIN, sử dụng Pandas merge.")
    df_student = pd.read_sql_query("SELECT * FROM student;", conn)
    df_course = pd.read_sql_query("SELECT * FROM course;", conn)
    display(pd.merge(df_student, df_course, left_on='course_id', right_on='id', how='outer'))


---

### 2. Cập nhật và xóa bản ghi
- Điền các `course_id` còn thiếu (lựa chọn môn 26: Tin học làm môn bổ sung cho những bạn chưa có).
- Xóa học sinh có mã môn học không thuộc bảng course.

In [ ]:
# Cập nhật course_id còn thiếu bằng 1 mã môn trong bảng course (Giả sử chọn mã 26 - Tin học)
conn.execute("UPDATE student SET course_id = 26 WHERE course_id IS NULL;")

# Loại bỏ những bản ghi tham gia môn học không tồn tại trong bảng course
conn.execute("DELETE FROM student WHERE course_id NOT IN (SELECT id FROM course);")
conn.commit()

# Xem kết quả sau khi cập nhật
print("Dữ liệu bảng student sau khi cập nhật:")
display(pd.read_sql_query("SELECT * FROM student", conn))


**2a. Tổng số sinh viên, điểm trung bình của từng lớp.**

In [ ]:
query_2a = '''
SELECT class as 'Lớp', 
       COUNT(student_id) as 'Tổng số sinh viên', 
       AVG(score) as 'Điểm TB'
FROM student
GROUP BY class;
'''
display(pd.read_sql_query(query_2a, conn))


**2b. Tổng số sinh viên, điểm trung bình của từng môn học.**

In [ ]:
query_2b = '''
SELECT c.course_name as 'Môn học',
       COUNT(s.student_id) as 'Tổng số sinh viên',
       AVG(s.score) as 'Điểm TB'
FROM student s
JOIN course c ON s.course_id = c.id
GROUP BY c.id, c.course_name;
'''
display(pd.read_sql_query(query_2b, conn))


**2c. Phân loại thi đua theo số điểm của từng môn học:**

In [ ]:
query_2c = '''
SELECT c.course_name as 'Môn học',
       AVG(s.score) as avg_score,
       CASE 
           WHEN AVG(s.score) >= 9.0 THEN 'Xuất sắc'
           WHEN AVG(s.score) >= 6.0 AND AVG(s.score) <= 8.9 THEN 'Tốt'
           WHEN AVG(s.score) < 6.0 THEN 'Kém'
       END as 'Phân loại'
FROM student s
JOIN course c ON s.course_id = c.id
GROUP BY c.id, c.course_name;
'''
display(pd.read_sql_query(query_2c, conn))


---

### 3. Xếp hạng sinh viên và hiển thị Top/Bottom 3

In [ ]:
# Xếp hạng sử dụng Window Functions trong SQL (RANK())

query_rank_score = '''
SELECT student_id, name, score, 
       RANK() OVER(ORDER BY score DESC) as rank_diem
FROM student;
'''

query_rank_class = '''
SELECT student_id, name, class, score,
       RANK() OVER(PARTITION BY class ORDER BY score DESC) as rank_theo_lop
FROM student;
'''

query_rank_course = '''
SELECT s.student_id, s.name, c.course_name, s.score,
       RANK() OVER(PARTITION BY s.course_id ORDER BY s.score DESC) as rank_theo_mon
FROM student s
JOIN course c ON s.course_id = c.id;
'''

print("--- Hạng theo điểm số ---")
df_rank_score = pd.read_sql_query(query_rank_score, conn)
display(df_rank_score)

print("\n=> TOP 3 Cao nhất (Theo điểm):")
display(df_rank_score.head(3))
print("=> TOP 3 Thấp nhất (Theo điểm):")
display(df_rank_score.tail(3))

print("\n\n--- Hạng theo lớp học ---")
df_rank_class = pd.read_sql_query(query_rank_class, conn)
display(df_rank_class)

print("\n--- Hạng theo môn học ---")
df_rank_course = pd.read_sql_query(query_rank_course, conn)
display(df_rank_course)


*(Mặc dù có Top/Bottom theo Lớp và Môn, phần in trên tập trung Top/Bottom theo điểm chung để tóm tắt. Bạn hoàn toàn có thể filter DataFrame cho Top/Bottom tương tự)*

---

### 4. Bổ sung trường `graduation_date`

Xác định thời gian tốt nghiệp bằng `thời gian hiện tại` cộng với `số hạng` của bạn đó (tính theo ngày).

In [ ]:
# Thêm cột graduation_date
conn.execute("ALTER TABLE student ADD COLUMN graduation_date DATETIME;")

# Ta phải kết hợp UPDATE với một mệnh đề hoặc CTE để cập nhật hạng của từng học sinh:
update_query = '''
WITH RankedStudents AS (
    SELECT student_id, RANK() OVER(ORDER BY score DESC) as rank_score
    FROM student
)
UPDATE student
SET graduation_date = DATETIME('now', 'localtime', '+' || (SELECT rank_score FROM RankedStudents WHERE RankedStudents.student_id = student.student_id) || ' days')
WHERE student_id IN (SELECT student_id FROM RankedStudents);
'''
conn.execute(update_query)
conn.commit()

# Hiển thị lại bảng student cuối cùng
display(pd.read_sql_query("SELECT student_id, name, class, score, graduation_date FROM student ORDER BY score DESC", conn))
